In [ ]:
# Criando DataFrame com os dados reais coletados acima
data_real = {
    "Formato": [f.upper() for f in formats],
    "Tamanho_MB": [tamanhos[f] for f in formats],
    "Escrita_s": [tempos_escrita[f] for f in formats],
    "Leitura_s": [tempos_leitura[f] if tempos_leitura[f] is not None else 0 for f in formats]
}
df_grafico = pd.DataFrame(data_real)

# Gráfico 1: Tamanho
plt.figure(figsize=(7,4))
plt.bar(df_grafico["Formato"], df_grafico["Tamanho_MB"], color="crimson")
plt.title("Tamanho dos Arquivos Gerados por Formato")
plt.ylabel("Tamanho (MB)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Gráfico 2: Tempos
plt.figure(figsize=(7,4))
bar_width = 0.35
x = range(len(df_grafico))
plt.bar(x, df_grafico["Escrita_s"], width=bar_width, label="Escrita (s)", color="steelblue")
plt.bar([p + bar_width for p in x], df_grafico["Leitura_s"], width=bar_width, label="Leitura (s)", color="orange")
plt.xticks([p + bar_width/2 for p in x], df_grafico["Formato"])
plt.title("Tempo de Escrita e Leitura por Formato")
plt.ylabel("Tempo (s)")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Criando DataFrame com os dados reais coletados acima
data_real = {
    "Formato": [f.upper() for f in formats],
    "Tamanho_MB": [tamanhos[f] for f in formats],
    "Escrita_s": [tempos_escrita[f] for f in formats],
    "Leitura_s": [tempos_leitura[f] if tempos_leitura[f] is not None else 0 for f in formats]
}
df_grafico = pd.DataFrame(data_real)

# Gráfico 1: Tamanho
plt.figure(figsize=(7,4))
plt.bar(df_grafico["Formato"], df_grafico["Tamanho_MB"], color="crimson")
plt.title("Tamanho dos Arquivos Gerados por Formato")
plt.ylabel("Tamanho (MB)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Gráfico 2: Tempos
plt.figure(figsize=(7,4))
bar_width = 0.35
x = range(len(df_grafico))
plt.bar(x, df_grafico["Escrita_s"], width=bar_width, label="Escrita (s)", color="steelblue")
plt.bar([p + bar_width for p in x], df_grafico["Leitura_s"], width=bar_width, label="Leitura (s)", color="orange")
plt.xticks([p + bar_width/2 for p in x], df_grafico["Formato"])
plt.title("Tempo de Escrita e Leitura por Formato")
plt.ylabel("Tempo (s)")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import time, os, json
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt
import pandas as pd

# =====================================================
# Configuração da Sessão Spark
# =====================================================
spark = (
    SparkSession.builder
    .appName("Benchmark-Formats")
    .master("local[*]")
    .config("spark.executor.memory", "1g")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    # Nota: Se der erro de SSL, mude a linha abaixo para "false"
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.committer.name", "directory")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

# =====================================================
# Carregando o Dataset base
# =====================================================
schema = "raceId INT, driverId INT, lap INT, position INT, time STRING, milliseconds INT"
df = spark.read.csv("/home/jovyan/data/lap_times.csv", header=True, schema=schema)
print("Linhas carregadas do CSV local:", df.count())

def medir_leitura(fmt, path):
    inicio = time.time()
    if fmt == "csv":
        df_res = spark.read.option("header", True).csv(path)
    elif fmt == "json":
        df_res = spark.read.json(path)
    elif fmt == "parquet":
        df_res = spark.read.parquet(path)
    elif fmt == "avro":
        df_res = spark.read.format("avro").option("ignoreExtension", "true").load(path)
    elif fmt == "orc":
        df_res = spark.read.orc(path)
    linhas = df_res.count()
    fim = time.time()
    return fim - inicio, linhas

# =====================================================
# Executando Escrita no MinIO
# =====================================================
base = "s3a://datalake/lap_times_"
formats = ["csv", "json", "parquet", "avro", "orc"]
tempos_escrita = {}

for fmt in formats:
    path = base + fmt
    print(f"Gravando {fmt.upper()} -> {path}")
    inicio = time.time()
    if fmt == "csv":
        df.write.mode("overwrite").option("header", True).csv(path)
    elif fmt == "json":
        df.write.mode("overwrite").json(path)
    elif fmt == "parquet":
        df.write.mode("overwrite").parquet(path)
    elif fmt == "avro":
        df.write.mode("overwrite").format("avro").save(path)
    elif fmt == "orc":
        df.write.mode("overwrite").orc(path)
    fim = time.time()
    tempos_escrita[fmt] = round(fim - inicio, 2)
    print(f"→ Tempo de escrita {fmt.upper()}: {tempos_escrita[fmt]}s")

# =====================================================
# Calculando Tamanhos dos Arquivos
# =====================================================
print("\nCalculando tamanhos no MinIO...")
!mc alias set local http://minio:9000 minioadmin minioadmin >/dev/null

tamanhos = {}
for fmt in formats:
    info = os.popen(f"mc du --json local/datalake/lap_times_{fmt}/").read()
    try:
        tamanho = float(json.loads(info)["size"]) / (1024 * 1024)
    except:
        tamanho = 0
    tamanhos[fmt] = round(tamanho, 2)

# =====================================================
# Executando Leitura e Benchmark
# =====================================================
print("\nIniciando testes de leitura...")
tempos_leitura = {}
linhas_lidas = {}

for fmt in formats:
    path = base + fmt
    try:
        tempo, linhas = medir_leitura(fmt, path)
        tempos_leitura[fmt] = round(tempo, 2)
        linhas_lidas[fmt] = linhas
        print(f"{fmt.upper()}: {linhas} linhas em {tempo:.2f}s")
    except Exception as e:
        tempos_leitura[fmt] = None
        linhas_lidas[fmt] = 0
        print(f"Erro ao ler {fmt}: {e}")

# =====================================================
# Resultado Consolidado em Tabela
# =====================================================
print("\n=== RESULTADO FINAL ===")
print(f"{'Formato':<10} {'Tamanho (MB)':>12} {'Escrita (s)':>12} {'Leitura (s)':>12} {'Linhas':>10}")
for fmt in formats:
    t_leit = tempos_leitura[fmt] if tempos_leitura[fmt] is not None else 0
    print(f"{fmt.upper():<10} {tamanhos[fmt]:>12.2f} {tempos_escrita[fmt]:>12.2f} {t_leit:>12.2f} {linhas_lidas[fmt]:>10}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 18:42:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Linhas carregadas do CSV local: 872521
Gravando CSV -> s3a://datalake/lap_times_csv
